#### ML Flow Experiment Tracking

##### Importing Libraires and Loading Data

In [18]:
import numpy as np
import pandas as pd
import mlflow
from mlflow import MlflowClient
import mlflow.xgboost
from pathlib import Path

In [5]:
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

In [10]:
project_root = Path('.').resolve().parent
fig_dir = project_root/'reports'/'figures'

mlflow.set_tracking_uri(f'sqlite:///{project_root/"mlflow.db"}')
mlflow.set_experiment('london-price-model')

<Experiment: artifact_location=('file:///C:/Users/Abdul Qudus/Documents/Data '
 'Portfolio/london-property-intelligence/notebooks/mlruns/1'), creation_time=1784394607651, experiment_id='1', last_update_time=1784394607651, lifecycle_stage='active', name='london-price-model', tags={}, trace_location=None, workspace='default'>

In [11]:
# Loading Data and Splitting

df = pd.read_parquet(project_root/'data'/'processed'/'london_features.parquet')

train = df[df['year'] <= 2023].copy()
test = df[df['year'] == 2024].copy()

In [13]:
# Target Encoding of Postcode District

# Global fallback: average log-price across all training rows
global_mean = train['log_price'].mean()

# Per-district mean log-price and how many training sales each district had
stats = train.groupby('postcode_district')['log_price'].agg(['mean', 'count'])

# Smoothing: districts with few sales get pulled toward the global mean
smooth = 20
encoding = (stats['mean'] * stats['count'] + global_mean * smooth) / (stats['count'] + smooth)

In [14]:
# Apply the Training encoding to both splits, unseen districts fall back to global mean

for part in (train, test):
    part['district_te'] = part['postcode_district'].map(encoding).fillna(global_mean)

# Impute the two columns with missing values

impute_medians = {}
for col in ['numberrooms', 'age_year']:
    impute_medians[col] = train[col].median()
    for part in (train, test):
        part[col] = part[col].fillna(impute_medians[col])

In [15]:
train.columns

Index(['price', 'log_price', 'postcode_district', 'lad23cd', 'log_tfarea',
       'numberrooms', 'age_year', 'energy_current', 'energy_potential', 'lat',
       'lon', 'zone', 'imd_index', 'log_income', 'dist_station',
       'propertytype_D', 'propertytype_F', 'propertytype_S', 'propertytype_T',
       'duration_F', 'duration_L', 'duration_U', 'year', 'month',
       'district_te'],
      dtype='object')

In [16]:
price_features = [
    'log_tfarea', 'numberrooms', 'age_year',
    'lat', 'lon', 'zone', 'imd_index', 'log_income', 'dist_station',
    'propertytype_D', 'propertytype_F', 'propertytype_S', 'propertytype_T',
    'duration_F', 'duration_L',
    'year', 'month', 'district_te'
]

##### ML Flow Logged Training Run

In [17]:
# Tuned XGBoost Configuration

params = {
    'n_estimators': 1400,
    'max_depth': 11,
    'learning_rate': 0.03,
    'min_child_weight': 30,
    'reg_lambda': 1.5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'tree_method': 'hist'
}

In [19]:
# One MLFlow Run

with mlflow.start_run(run_name='xgboost-tuned') as run:

   # Parameters
    mlflow.log_params(params)
    mlflow.log_param('n_features', len(price_features))
    mlflow.log_param('train_through_year', 2023)

    # Model Training
    model = XGBRegressor(random_state=69, n_jobs=-1, **params)
    model.fit(train[price_features], train['log_price'])

    pred_log = model.predict(test[price_features])
    pred_gbp = 10 ** pred_log
    true_gbp = 10 ** test['log_price']

    # Metrics
    mlflow.log_metric('r2_log', r2_score(test['log_price'], pred_log))
    mlflow.log_metric('r2_gbp', r2_score(true_gbp, pred_gbp))
    mlflow.log_metric('mae_gbp', mean_absolute_error(true_gbp, pred_gbp))
    mlflow.log_metric('median_ae_gbp', float(np.median(np.abs(true_gbp - pred_gbp))))

    mlflow.xgboost.log_model(
        model,
        name='model',
        input_example=test[price_features].head(5)
    )

    print('run_id:', run.info.run_id)

C:\Users\Abdul Qudus\anaconda3\envs\london-property\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


run_id: 643ab624b219478aa0ecd1182455494c


In [24]:
# Client for Registry Operations

client = MlflowClient()

versions = client.search_model_versions('name="london-price"')
latest =  max(int(v.version) for v in versions)

In [25]:
client.set_registered_model_alias('london-price', 'champion', latest)

In [27]:
loaded = mlflow.xgboost.load_model("models:/london-price@champion")